# Full Fine-Tuning local via Hugging Face (alternativa multiplataforma ao MLX)

**Ahirton Lopes · Fine-Tuning Toolkit**
**Artefato de Demo - Módulo 4.4, alternativa pra quem não tem Mac Apple Silicon**

O Módulo 4.4 desta disciplina treina full fine-tuning local de verdade via `mlx-lm` (`--fine-tune-type full`), que só roda em Apple Silicon. Este notebook faz a mesma coisa - full fine-tuning genuíno, cem por cento dos parâmetros treináveis, sem LoRA - com o stack Hugging Face (`transformers` + `trl` + `bitsandbytes`), que roda em qualquer GPU CUDA, inclusive a T4 gratuita do Colab.

**Diferença importante e honesta em relação ao Módulo 4.4**: o modelo usado lá (Gemma 4 E2B, ~5,12 bilhões de parâmetros reais, confirmado via API da Hugging Face) não cabe em full fine-tuning numa T4 de 16GB - a conta real de memória (peso mestre + gradiente + estado do otimizador) fica entre 27GB e 74GB dependendo da configuração, bem acima do limite físico da GPU gratuita. Não existe otimização isolada (quantização, gradient checkpointing, otimizador 8 bits) que resolva isso sozinha: quantizar os pesos em 4 bits, como o QLoRA faz, impede justamente treinar os pesos originais - é por isso que QLoRA treina só um adaptador por cima, não os pesos de verdade. Este notebook usa a família **Qwen3/Qwen2.5**, bem menor (entre ~0,75 e ~2,03 bilhões de parâmetros reais, ver Passo 0), pra dar uma demonstração de full fine-tuning genuíno que cabe de verdade numa T4 grátis - mesma mecânica, mesmo dataset real, escala de modelo diferente.

**Como rodar**: Menu **Runtime > Change runtime type > T4 GPU**, depois **Runtime > Run all**.

**MLX continua sendo o caminho oficial** desta disciplina - o Módulo 4.4 usa os números reais do MLX (Gemma 4 E2B, full fine-tuning das últimas 16 camadas) pra toda a comparação de trade-off contra LoRA nos Módulos 4.3 e 4.4. Este notebook é o caminho alternativo, pra quem não tem Mac com Apple Silicon e quer rodar full fine-tuning de verdade mesmo assim, com um modelo de escala que cabe na GPU gratuita, em vez de só documentar que não foi possível rodar.

**Status de validação (honesto)**: rodado de verdade numa GPU T4 real do Colab, do início ao fim, sem erro, com o modelo default (`Qwen/Qwen3-1.7B`, 1.720.574.976 parâmetros treináveis, otimizador `adamw_8bit`). Os outputs completos dessa execução real - célula por célula - estão salvos no próprio notebook, pra quem quiser conferir sem rodar nada. Resultado real:

- **Treino**: 20 passos (batch 1, mesmo orçamento do Módulo 4.4 real), 38 segundos de duração.
- **Loss de treino no passo final**: 1,444
- **Loss de validação no passo final**: 1,302
- **Acurácia média de token**: 75,0%
- **Pico real de memória de GPU**: 13,51GB - cabe na T4 de 16GB, com ~2,5GB de folga.
- **Comparação com o Módulo 4.4 (MLX, Gemma 4 E2B)**: lá, val loss final do full fine-tuning foi 0,612 (contra 1,302 aqui) - os dois modelos e frameworks não são comparáveis número a número (escala de modelo, dataset de pré-treino e otimizador diferentes), mas os dois confirmam a mesma conclusão qualitativa: full fine-tuning genuíno, mesmo com orçamento curto de treino (20 passos), já produz ajuste real e mensurável nesse dataset.
- **Teste do exemplo difícil (Passo 4)**: o modelo extraiu corretamente os três campos centrais, ignorando o distrator de propósito (`placa: TUV-4499`, `valor: 2310.75` - não a revisão antiga, `QRS-1122`, R$ 890,00). **Achado real, documentado sem maquiagem**: a resposta também trouxe campos extras que não existem no dataset de treino (`data`, `oficinas`, `data_pagamento`) e não fechou o JSON dentro do orçamento de `max_new_tokens=100` - ver nota logo depois da célula do Passo 4. Isso não invalida o teste - o padrão de extração central foi aprendido corretamente -, mas mostra que 20 passos, suficientes pra travar o formato de saída no LoRA do Módulo 4.2 (adaptador pequeno, mudança cirúrgica), não bastaram pra travar o formato tão rigidamente num full fine-tuning de um modelo maior e mais "solto" por padrão.


In [1]:
!pip install -q -U transformers trl accelerate datasets bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.0 MB/s eta 0:00:00


## Passo 0 - Escolher o modelo (3 opções reais, verificadas)

Três modelos pequenos, licença Apache 2.0, sem necessidade de aprovar termo de uso na Hugging Face (`"gated": false` confirmado direto na API do Hub em 16/09/2026 pros três) - diferente de alternativas como Gemma 3 1B ou Llama 3.2 1B, que exigem login e aprovação manual de licença antes de baixar. Todos carregam com a mesma classe padrão (`AutoModelForCausalLM`) e o mesmo formato de dataset (`messages`) já usado no resto da disciplina, e todos têm `tie_word_embeddings=True` (o `embed_tokens` e o `lm_head` compartilham o mesmo peso em runtime).

| Modelo | Parâmetros treináveis reais (o que importa pra memória, não o nome comercial) | Memória estática estimada (peso + gradiente + otimizador) | Observação |
|---|---|---|---|
| `Qwen/Qwen3-1.7B` (default) | 1.720.574.976 | ~10,3GB com otimizador 8 bits, até ~27,5GB sem nenhuma otimização | Melhor qualidade dos três; cabe na T4 só com otimizador 8 bits + gradient checkpointing |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.310.340.608 | ~7,9GB a ~21,0GB | Sem modo "thinking" do Qwen3, mais simples de explicar em aula |
| `Qwen/Qwen3-0.6B` | 596.049.920 | ~3,6GB a ~9,5GB | Mais folga de memória - sobra espaço pra batch maior ou sequência mais longa; cabe até sem otimizador 8 bits |

A contagem "treináveis reais" acima já é a corrigida, não o número bruto que a API da Hugging Face mostra em `safetensors.total` (2.031.739.904 / 1.543.714.304 / 751.632.384) - esse número bruto conta a tabela de embedding duas vezes, porque o arquivo salvo no disco guarda `embed_tokens` e `lm_head` como tensores separados mesmo eles sendo o mesmo peso em runtime (`tie_word_embeddings=True`). A diferença é exatamente `vocab_size × hidden_size` (151.936 × 2048/1536/1024). Confirmado rodando de verdade no Passo 2 mais abaixo - `model.parameters()` bate com a contagem "treináveis reais" desta tabela, não com o número bruto da API. Mesmo tipo de pegadinha de nomenclatura do "E2B" do Gemma 4 usado no resto da disciplina, só que na direção oposta (lá o nome comercial subestima o tamanho real; aqui o número bruto da API superestima).

Trocar de modelo é só mudar `MODEL_ID` na célula abaixo e rodar tudo de novo - nenhuma outra célula deste notebook precisa mudar.


In [2]:
MODEL_ID = "Qwen/Qwen3-1.7B"  # troque para "Qwen/Qwen2.5-1.5B-Instruct" ou "Qwen/Qwen3-0.6B" se preferir

MODELOS_VALIDADOS = {
    # params_treinaveis = safetensors.total da API HF menos vocab_size x hidden_size
    # (embed_tokens/lm_head contados 2x no arquivo, tied em runtime - ver Passo 0)
    "Qwen/Qwen3-1.7B": {"params_treinaveis": 1_720_574_976, "usa_thinking": True},
    "Qwen/Qwen2.5-1.5B-Instruct": {"params_treinaveis": 1_310_340_608, "usa_thinking": False},
    "Qwen/Qwen3-0.6B": {"params_treinaveis": 596_049_920, "usa_thinking": True},
}


def validar_hiperparametros(model_id, max_steps, learning_rate):
    erros = []
    if model_id not in MODELOS_VALIDADOS:
        erros.append(
            f"MODEL_ID fora da lista validada desta disciplina: {model_id!r}. "
            f"Opções validadas: {sorted(MODELOS_VALIDADOS)}"
        )
    if not isinstance(max_steps, int) or not (1 <= max_steps <= 1000):
        erros.append(f"max_steps fora da faixa 1-1000: {max_steps}")
    if not (1e-7 <= learning_rate <= 1e-2):
        erros.append(f"learning_rate fora da faixa 1e-7 a 1e-2: {learning_rate}")
    if erros:
        raise ValueError("Hiperparâmetros inválidos:\n  " + "\n  ".join(erros))
    return True


MAX_STEPS = 20        # mesmo orçamento do Módulo 4.4 real (MLX, --iters 20)
LEARNING_RATE = 1e-5  # mesmo valor do Módulo 4.4 real (MLX, --learning-rate 1e-5)

validar_hiperparametros(MODEL_ID, MAX_STEPS, LEARNING_RATE)
USA_THINKING = MODELOS_VALIDADOS[MODEL_ID]["usa_thinking"]
print(f"Hiperparâmetros validados. Modelo: {MODEL_ID} ({MODELOS_VALIDADOS[MODEL_ID]['params_treinaveis']:,} parâmetros treináveis)")


Hiperparâmetros validados. Modelo: Qwen/Qwen3-1.7B (1,720,574,976 parâmetros treináveis)


**Nota sobre o parâmetro `enable_thinking`**: os modelos Qwen3 (`Qwen3-1.7B`, `Qwen3-0.6B`) são híbridos "thinking/non-thinking" - o chat template deles aceita `enable_thinking=True` ou `False`. Pra fine-tuning supervisionado de domínio, com respostas diretas em JSON e sem raciocínio exposto, a prática recomendada é `enable_thinking=False`. O `Qwen2.5-1.5B-Instruct` não tem esse modo e não aceita esse parâmetro - a função `aplicar_chat_template` abaixo trata os dois casos automaticamente a partir da escolha feita no Passo 0, então o resto do notebook funciona sem alteração com qualquer um dos três modelos.

**Achado real testando isso**: mesmo com `enable_thinking=False`, o template do Qwen3 insere um bloco `<think>\n\n</think>\n\n` vazio antes da resposta - não é um bug, é o formato oficial do modelo pra marcar explicitamente "sem raciocínio" (confirmado rodando de verdade, não é suposição). Não precisa remover isso na mão: o modelo aprende a reproduzir esse mesmo padrão vazio durante o treino, porque ele também está nos dados de treino gerados pela mesma função.


In [3]:
def aplicar_chat_template(tokenizer, mensagens, add_generation_prompt):
    if USA_THINKING:
        return tokenizer.apply_chat_template(
            mensagens, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False,
        )
    return tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=add_generation_prompt,
    )


## Passo 1 - Carregar o mesmo dataset real do Módulo 2.2

157 exemplos de treino, 30 de validação, 13 de teste - os mesmos 200 exemplos reais (Amplitude Auto + Saúde Empresarial) que o Módulo 3 subiu pro Vertex AI, o Módulo 4.2 treinou local via MLX/LoRA, e o `colab-lora-training-notebook.ipynb` (Módulo 4.2, alternativa não-Mac) já usa. Embutido aqui, sem depender do resto do repositório, pra abrir e rodar sem precisar clonar nada.


In [4]:
import json

TRAIN_JSONL = r"""{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Vinicius Augusto Teixeira Placa do veiculo: YHN-3392 Data do sinistro: 20/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Vinicius Augusto Teixeira Placa: YHN-3392 Data do atendimento: 20/02/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Vinicius Augusto Teixeira Placa do veiculo: YHN-3392 Data: 20/02/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Vinicius Augusto Teixeira Placa do carro: YHN-3392 Data do orcamento: 20/02/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.780,90"},{"role":"assistant","content":"{\"segurado\":\"Vinicius Augusto Teixeira\",\"placa\":\"YHN-3392\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Vinicius Augusto Teixeira Procedimento: consulta geriatrica Data do atendimento: 20/02/2026 Valor cobrado: R$ 3.780,90"},{"role":"assistant","content":"{\"beneficiario\":\"Vinicius Augusto Teixeira\",\"procedimento\":\"consulta geriatrica\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Vinicius Augusto Teixeira Procedimento realizado: consulta geriatrica Data: 20/02/2026 Valor total: R$ 3.780,90"},{"role":"assistant","content":"{\"beneficiario\":\"Vinicius Augusto Teixeira\",\"procedimento\":\"consulta geriatrica\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Vinicius Augusto Teixeira Procedimento: consulta geriatrica Data da consulta: 20/02/2026 Valor cobrado: R$ 3.780,90"},{"role":"assistant","content":"{\"beneficiario\":\"Vinicius Augusto Teixeira\",\"procedimento\":\"consulta geriatrica\",\"valor\":3780.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Diego Cristina Vieira Placa do veiculo: ASD-6673 Data do sinistro: 05/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Diego Cristina Vieira Placa: ASD-6673 Data do atendimento: 05/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Diego Cristina Vieira Placa do veiculo: ASD-6673 Data: 05/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Diego Cristina Vieira Placa do carro: ASD-6673 Data do orcamento: 05/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Diego Cristina Vieira Placa do veiculo: ASD-6673 Data do servico: 05/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 3.870,00"},{"role":"assistant","content":"{\"segurado\":\"Diego Cristina Vieira\",\"placa\":\"ASD-6673\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Diego Cristina Vieira Procedimento: exame de urina completo Data do atendimento: 05/04/2026 Valor cobrado: R$ 3.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Diego Cristina Vieira\",\"procedimento\":\"exame de urina completo\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Diego Cristina Vieira Procedimento realizado: exame de urina completo Data: 05/04/2026 Valor total: R$ 3.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Diego Cristina Vieira\",\"procedimento\":\"exame de urina completo\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Diego Cristina Vieira Procedimento: exame de urina completo Data da consulta: 05/04/2026 Valor cobrado: R$ 3.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Diego Cristina Vieira\",\"procedimento\":\"exame de urina completo\",\"valor\":3870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Gustavo Souza Albuquerque Placa do veiculo: TGB-2286 Data do sinistro: 10/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Gustavo Souza Albuquerque Placa: TGB-2286 Data do atendimento: 10/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Gustavo Souza Albuquerque Placa do veiculo: TGB-2286 Data: 10/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Gustavo Souza Albuquerque Placa do carro: TGB-2286 Data do orcamento: 10/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Gustavo Souza Albuquerque Placa do veiculo: TGB-2286 Data do servico: 10/06/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.590,00"},{"role":"assistant","content":"{\"segurado\":\"Gustavo Souza Albuquerque\",\"placa\":\"TGB-2286\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Gustavo Souza Albuquerque Procedimento: consulta ortopedica Data do atendimento: 10/06/2026 Valor cobrado: R$ 1.590,00"},{"role":"assistant","content":"{\"beneficiario\":\"Gustavo Souza Albuquerque\",\"procedimento\":\"consulta ortopedica\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Gustavo Souza Albuquerque Procedimento realizado: consulta ortopedica Data: 10/06/2026 Valor total: R$ 1.590,00"},{"role":"assistant","content":"{\"beneficiario\":\"Gustavo Souza Albuquerque\",\"procedimento\":\"consulta ortopedica\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Gustavo Souza Albuquerque Procedimento: consulta ortopedica Data da consulta: 10/06/2026 Valor cobrado: R$ 1.590,00"},{"role":"assistant","content":"{\"beneficiario\":\"Gustavo Souza Albuquerque\",\"procedimento\":\"consulta ortopedica\",\"valor\":1590}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Renata Tavares Siqueira Placa do veiculo: CDE-3396 Data do sinistro: 02/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Renata Tavares Siqueira Placa: CDE-3396 Data do atendimento: 02/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Renata Tavares Siqueira Placa do veiculo: CDE-3396 Data: 02/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Renata Tavares Siqueira Placa do carro: CDE-3396 Data do orcamento: 02/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Renata Tavares Siqueira Placa do veiculo: CDE-3396 Data do servico: 02/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 3.980,60"},{"role":"assistant","content":"{\"segurado\":\"Renata Tavares Siqueira\",\"placa\":\"CDE-3396\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Renata Tavares Siqueira Procedimento: consulta dermatologica Data do atendimento: 02/04/2026 Valor cobrado: R$ 3.980,60"},{"role":"assistant","content":"{\"beneficiario\":\"Renata Tavares Siqueira\",\"procedimento\":\"consulta dermatologica\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Renata Tavares Siqueira Procedimento realizado: consulta dermatologica Data: 02/04/2026 Valor total: R$ 3.980,60"},{"role":"assistant","content":"{\"beneficiario\":\"Renata Tavares Siqueira\",\"procedimento\":\"consulta dermatologica\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Renata Tavares Siqueira Procedimento: consulta dermatologica Data da consulta: 02/04/2026 Valor cobrado: R$ 3.980,60"},{"role":"assistant","content":"{\"beneficiario\":\"Renata Tavares Siqueira\",\"procedimento\":\"consulta dermatologica\",\"valor\":3980.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Bruno Luiz Bessa Placa do veiculo: PLW-7765 Data do sinistro: 16/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Bruno Luiz Bessa Placa: PLW-7765 Data do atendimento: 16/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Bruno Luiz Bessa Placa do veiculo: PLW-7765 Data: 16/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Bruno Luiz Bessa Placa do carro: PLW-7765 Data do orcamento: 16/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Bruno Luiz Bessa Placa do veiculo: PLW-7765 Data do servico: 16/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"segurado\":\"Bruno Luiz Bessa\",\"placa\":\"PLW-7765\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Bruno Luiz Bessa Procedimento: sessao de psicoterapia Data do atendimento: 16/04/2026 Valor cobrado: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Bruno Luiz Bessa Procedimento realizado: sessao de psicoterapia Data: 16/04/2026 Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Bruno Luiz Bessa Procedimento: sessao de psicoterapia Data da consulta: 16/04/2026 Valor cobrado: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Bruno Luiz Bessa Procedimento: sessao de psicoterapia Data do atendimento: 16/04/2026 Valor total: R$ 1.395,00"},{"role":"assistant","content":"{\"beneficiario\":\"Bruno Luiz Bessa\",\"procedimento\":\"sessao de psicoterapia\",\"valor\":1395}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Juliana Henrique Barros Placa do veiculo: HGF-3391 Data do sinistro: 22/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Juliana Henrique Barros Placa: HGF-3391 Data do atendimento: 22/07/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Juliana Henrique Barros Placa do veiculo: HGF-3391 Data: 22/07/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Juliana Henrique Barros Placa do carro: HGF-3391 Data do orcamento: 22/07/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Juliana Henrique Barros Placa do veiculo: HGF-3391 Data do servico: 22/07/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"segurado\":\"Juliana Henrique Barros\",\"placa\":\"HGF-3391\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Juliana Henrique Barros Procedimento: consulta urologica Data do atendimento: 22/07/2026 Valor cobrado: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Juliana Henrique Barros Procedimento realizado: consulta urologica Data: 22/07/2026 Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Juliana Henrique Barros Procedimento: consulta urologica Data da consulta: 22/07/2026 Valor cobrado: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Juliana Henrique Barros Procedimento: consulta urologica Data do atendimento: 22/07/2026 Valor total: R$ 4.950,00"},{"role":"assistant","content":"{\"beneficiario\":\"Juliana Henrique Barros\",\"procedimento\":\"consulta urologica\",\"valor\":4950}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Patricia Pereira Godoy Placa do veiculo: OLP-1122 Data do sinistro: 09/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Patricia Pereira Godoy Placa: OLP-1122 Data do atendimento: 09/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Patricia Pereira Godoy Placa do veiculo: OLP-1122 Data: 09/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Patricia Pereira Godoy Placa do carro: OLP-1122 Data do orcamento: 09/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Patricia Pereira Godoy Placa do veiculo: OLP-1122 Data do servico: 09/04/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Patricia Pereira Godoy Placa: OLP-1122 Data do atendimento: 09/04/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"segurado\":\"Patricia Pereira Godoy\",\"placa\":\"OLP-1122\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Patricia Pereira Godoy Procedimento: consulta neurologica Data do atendimento: 09/04/2026 Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Patricia Pereira Godoy Procedimento realizado: consulta neurologica Data: 09/04/2026 Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Patricia Pereira Godoy Procedimento: consulta neurologica Data da consulta: 09/04/2026 Valor cobrado: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Patricia Pereira Godoy Procedimento: consulta neurologica Data do atendimento: 09/04/2026 Valor total: R$ 1.540,00"},{"role":"assistant","content":"{\"beneficiario\":\"Patricia Pereira Godoy\",\"procedimento\":\"consulta neurologica\",\"valor\":1540}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Thiago Augusto Barbosa Placa do veiculo: ERT-8873 Data do sinistro: 02/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Thiago Augusto Barbosa Placa: ERT-8873 Data do atendimento: 02/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Thiago Augusto Barbosa Placa do veiculo: ERT-8873 Data: 02/05/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Thiago Augusto Barbosa Placa do carro: ERT-8873 Data do orcamento: 02/05/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Thiago Augusto Barbosa Placa do veiculo: ERT-8873 Data do servico: 02/05/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Thiago Augusto Barbosa Placa: ERT-8873 Data do atendimento: 02/05/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"segurado\":\"Thiago Augusto Barbosa\",\"placa\":\"ERT-8873\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Thiago Augusto Barbosa Procedimento: consulta ginecologica Data do atendimento: 02/05/2026 Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Thiago Augusto Barbosa Procedimento realizado: consulta ginecologica Data: 02/05/2026 Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Thiago Augusto Barbosa Procedimento: consulta ginecologica Data da consulta: 02/05/2026 Valor cobrado: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Thiago Augusto Barbosa Procedimento: consulta ginecologica Data do atendimento: 02/05/2026 Valor total: R$ 4.430,00"},{"role":"assistant","content":"{\"beneficiario\":\"Thiago Augusto Barbosa\",\"procedimento\":\"consulta ginecologica\",\"valor\":4430}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Beatriz Luiz Ramalho Placa do veiculo: XSW-6652 Data do sinistro: 04/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Beatriz Luiz Ramalho Placa: XSW-6652 Data do atendimento: 04/02/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Beatriz Luiz Ramalho Placa do veiculo: XSW-6652 Data: 04/02/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Beatriz Luiz Ramalho Placa do carro: XSW-6652 Data do orcamento: 04/02/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Beatriz Luiz Ramalho Placa do veiculo: XSW-6652 Data do servico: 04/02/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Beatriz Luiz Ramalho Placa: XSW-6652 Data do atendimento: 04/02/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"segurado\":\"Beatriz Luiz Ramalho\",\"placa\":\"XSW-6652\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Beatriz Luiz Ramalho Procedimento: exame de eletrocardiograma Data do atendimento: 04/02/2026 Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Beatriz Luiz Ramalho Procedimento realizado: exame de eletrocardiograma Data: 04/02/2026 Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Beatriz Luiz Ramalho Procedimento: exame de eletrocardiograma Data da consulta: 04/02/2026 Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Beatriz Luiz Ramalho Procedimento: exame de eletrocardiograma Data do atendimento: 04/02/2026 Valor total: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Beatriz Luiz Ramalho Procedimento realizado: exame de eletrocardiograma Data: 04/02/2026 Valor cobrado: R$ 4.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Beatriz Luiz Ramalho\",\"procedimento\":\"exame de eletrocardiograma\",\"valor\":4120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Camila Costa Ribeiro Placa do veiculo: AZS-6617 Data do sinistro: 21/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Camila Costa Ribeiro Placa: AZS-6617 Data do atendimento: 21/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Camila Costa Ribeiro Placa do veiculo: AZS-6617 Data: 21/05/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Camila Costa Ribeiro Placa do carro: AZS-6617 Data do orcamento: 21/05/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Camila Costa Ribeiro Placa do veiculo: AZS-6617 Data do servico: 21/05/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Camila Costa Ribeiro Placa: AZS-6617 Data do atendimento: 21/05/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"segurado\":\"Camila Costa Ribeiro\",\"placa\":\"AZS-6617\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Camila Costa Ribeiro Procedimento: sessao de fonoterapia Data do atendimento: 21/05/2026 Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Camila Costa Ribeiro Procedimento realizado: sessao de fonoterapia Data: 21/05/2026 Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Camila Costa Ribeiro Procedimento: sessao de fonoterapia Data da consulta: 21/05/2026 Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Camila Costa Ribeiro Procedimento: sessao de fonoterapia Data do atendimento: 21/05/2026 Valor total: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Camila Costa Ribeiro Procedimento realizado: sessao de fonoterapia Data: 21/05/2026 Valor cobrado: R$ 1.780,50"},{"role":"assistant","content":"{\"beneficiario\":\"Camila Costa Ribeiro\",\"procedimento\":\"sessao de fonoterapia\",\"valor\":1780.5}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Eduardo Moreira Duarte Placa do veiculo: TGB-9958 Data do sinistro: 07/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Eduardo Moreira Duarte Placa: TGB-9958 Data do atendimento: 07/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Eduardo Moreira Duarte Placa do veiculo: TGB-9958 Data: 07/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Eduardo Moreira Duarte Placa do carro: TGB-9958 Data do orcamento: 07/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Eduardo Moreira Duarte Placa do veiculo: TGB-9958 Data do servico: 07/06/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Eduardo Moreira Duarte Placa: TGB-9958 Data do atendimento: 07/06/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"segurado\":\"Eduardo Moreira Duarte\",\"placa\":\"TGB-9958\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Eduardo Moreira Duarte Procedimento: fisioterapia ortopedica Data do atendimento: 07/06/2026 Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Eduardo Moreira Duarte Procedimento realizado: fisioterapia ortopedica Data: 07/06/2026 Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Eduardo Moreira Duarte Procedimento: fisioterapia ortopedica Data da consulta: 07/06/2026 Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Eduardo Moreira Duarte Procedimento: fisioterapia ortopedica Data do atendimento: 07/06/2026 Valor total: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Eduardo Moreira Duarte Procedimento realizado: fisioterapia ortopedica Data: 07/06/2026 Valor cobrado: R$ 3.660,00"},{"role":"assistant","content":"{\"beneficiario\":\"Eduardo Moreira Duarte\",\"procedimento\":\"fisioterapia ortopedica\",\"valor\":3660}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Fernanda Cesar Figueiredo Placa do veiculo: POI-7738 Data do sinistro: 28/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Fernanda Cesar Figueiredo Placa: POI-7738 Data do atendimento: 28/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Fernanda Cesar Figueiredo Placa do veiculo: POI-7738 Data: 28/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Fernanda Cesar Figueiredo Placa do carro: POI-7738 Data do orcamento: 28/03/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Fernanda Cesar Figueiredo Placa do veiculo: POI-7738 Data do servico: 28/03/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Fernanda Cesar Figueiredo Placa: POI-7738 Data do atendimento: 28/03/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"segurado\":\"Fernanda Cesar Figueiredo\",\"placa\":\"POI-7738\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Fernanda Cesar Figueiredo Procedimento: sessao de fonoaudiologia Data do atendimento: 28/03/2026 Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Fernanda Cesar Figueiredo Procedimento realizado: sessao de fonoaudiologia Data: 28/03/2026 Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Fernanda Cesar Figueiredo Procedimento: sessao de fonoaudiologia Data da consulta: 28/03/2026 Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Fernanda Cesar Figueiredo Procedimento: sessao de fonoaudiologia Data do atendimento: 28/03/2026 Valor total: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Fernanda Cesar Figueiredo Procedimento realizado: sessao de fonoaudiologia Data: 28/03/2026 Valor cobrado: R$ 5.120,00"},{"role":"assistant","content":"{\"beneficiario\":\"Fernanda Cesar Figueiredo\",\"procedimento\":\"sessao de fonoaudiologia\",\"valor\":5120}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Joaquim Ferreira Nunes Placa do veiculo: FGH-8852 Data do sinistro: 15/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Joaquim Ferreira Nunes Placa: FGH-8852 Data do atendimento: 15/07/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Joaquim Ferreira Nunes Placa do veiculo: FGH-8852 Data: 15/07/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Joaquim Ferreira Nunes Placa do carro: FGH-8852 Data do orcamento: 15/07/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Joaquim Ferreira Nunes Placa do veiculo: FGH-8852 Data do servico: 15/07/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Joaquim Ferreira Nunes Placa: FGH-8852 Data do atendimento: 15/07/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"segurado\":\"Joaquim Ferreira Nunes\",\"placa\":\"FGH-8852\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Joaquim Ferreira Nunes Procedimento: sessao de acupuntura Data do atendimento: 15/07/2026 Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Joaquim Ferreira Nunes Procedimento realizado: sessao de acupuntura Data: 15/07/2026 Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Joaquim Ferreira Nunes Procedimento: sessao de acupuntura Data da consulta: 15/07/2026 Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Joaquim Ferreira Nunes Procedimento: sessao de acupuntura Data do atendimento: 15/07/2026 Valor total: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Joaquim Ferreira Nunes Procedimento realizado: sessao de acupuntura Data: 15/07/2026 Valor cobrado: R$ 2.870,60"},{"role":"assistant","content":"{\"beneficiario\":\"Joaquim Ferreira Nunes\",\"procedimento\":\"sessao de acupuntura\",\"valor\":2870.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Larissa Lopes Guimaraes Placa do veiculo: GHJ-5529 Data do sinistro: 13/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Larissa Lopes Guimaraes Placa: GHJ-5529 Data do atendimento: 13/02/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Larissa Lopes Guimaraes Placa do veiculo: GHJ-5529 Data: 13/02/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Larissa Lopes Guimaraes Placa do carro: GHJ-5529 Data do orcamento: 13/02/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Larissa Lopes Guimaraes Placa do veiculo: GHJ-5529 Data do servico: 13/02/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Larissa Lopes Guimaraes Placa: GHJ-5529 Data do atendimento: 13/02/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"segurado\":\"Larissa Lopes Guimaraes\",\"placa\":\"GHJ-5529\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Larissa Lopes Guimaraes Procedimento: exame oftalmologico Data do atendimento: 13/02/2026 Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Larissa Lopes Guimaraes Procedimento realizado: exame oftalmologico Data: 13/02/2026 Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Larissa Lopes Guimaraes Procedimento: exame oftalmologico Data da consulta: 13/02/2026 Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Larissa Lopes Guimaraes Procedimento: exame oftalmologico Data do atendimento: 13/02/2026 Valor total: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Larissa Lopes Guimaraes Procedimento realizado: exame oftalmologico Data: 13/02/2026 Valor cobrado: R$ 2.210,00"},{"role":"assistant","content":"{\"beneficiario\":\"Larissa Lopes Guimaraes\",\"procedimento\":\"exame oftalmologico\",\"valor\":2210}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Marcos Nogueira Lima Placa do veiculo: MNB-3310 Data do sinistro: 25/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Marcos Nogueira Lima Placa: MNB-3310 Data do atendimento: 25/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Marcos Nogueira Lima Placa do veiculo: MNB-3310 Data: 25/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Marcos Nogueira Lima Placa do carro: MNB-3310 Data do orcamento: 25/03/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Marcos Nogueira Lima Placa do veiculo: MNB-3310 Data do servico: 25/03/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Marcos Nogueira Lima Placa: MNB-3310 Data do atendimento: 25/03/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"segurado\":\"Marcos Nogueira Lima\",\"placa\":\"MNB-3310\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Marcos Nogueira Lima Procedimento: exame de sangue completo Data do atendimento: 25/03/2026 Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Marcos Nogueira Lima Procedimento realizado: exame de sangue completo Data: 25/03/2026 Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Marcos Nogueira Lima Procedimento: exame de sangue completo Data da consulta: 25/03/2026 Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Marcos Nogueira Lima Procedimento: exame de sangue completo Data do atendimento: 25/03/2026 Valor total: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Marcos Nogueira Lima Procedimento realizado: exame de sangue completo Data: 25/03/2026 Valor cobrado: R$ 4.120,30"},{"role":"assistant","content":"{\"beneficiario\":\"Marcos Nogueira Lima\",\"procedimento\":\"exame de sangue completo\",\"valor\":4120.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Rafael Lopes Assuncao Placa do veiculo: CVB-1120 Data do sinistro: 12/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Rafael Lopes Assuncao Placa: CVB-1120 Data do atendimento: 12/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Rafael Lopes Assuncao Placa do veiculo: CVB-1120 Data: 12/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Rafael Lopes Assuncao Placa do carro: CVB-1120 Data do orcamento: 12/03/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nMECANICA HORIZONTE LTDA CNPJ 51.222.888/0001-19 Av. do Horizonte 640 Segurado: Rafael Lopes Assuncao Placa do veiculo: CVB-1120 Data do servico: 12/03/2026 Descricao: alinhamento e balanceamento apos colisao lateral Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO REPAROS UNIAO ME CNPJ 63.444.777/0001-28 Rua da Uniao 305 Nome do segurado: Rafael Lopes Assuncao Placa: CVB-1120 Data do atendimento: 12/03/2026 Servico: troca de para-brisa trincado Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"segurado\":\"Rafael Lopes Assuncao\",\"placa\":\"CVB-1120\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Rafael Lopes Assuncao Procedimento: exame de mamografia Data do atendimento: 12/03/2026 Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Rafael Lopes Assuncao Procedimento realizado: exame de mamografia Data: 12/03/2026 Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Rafael Lopes Assuncao Procedimento: exame de mamografia Data da consulta: 12/03/2026 Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA SAO RAFAEL CNPJ 84.111.222/0001-37 Rua Sao Rafael 512 Paciente/Beneficiario: Rafael Lopes Assuncao Procedimento: exame de mamografia Data do atendimento: 12/03/2026 Valor total: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nINSTITUTO SAUDE PLENA LTDA CNPJ 91.333.555/0001-08 Av. da Saude Plena 78 Beneficiario: Rafael Lopes Assuncao Procedimento realizado: exame de mamografia Data: 12/03/2026 Valor cobrado: R$ 5.640,00"},{"role":"assistant","content":"{\"beneficiario\":\"Rafael Lopes Assuncao\",\"procedimento\":\"exame de mamografia\",\"valor\":5640}"}]}"""

VALID_JSONL = r"""{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Rodrigo Braga Quintanilha Placa do veiculo: ZXC-2298 Data do sinistro: 29/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Rodrigo Braga Quintanilha Placa: ZXC-2298 Data do atendimento: 29/07/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Rodrigo Braga Quintanilha Placa do veiculo: ZXC-2298 Data: 29/07/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Rodrigo Braga Quintanilha Placa do carro: ZXC-2298 Data do orcamento: 29/07/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.660,40"},{"role":"assistant","content":"{\"segurado\":\"Rodrigo Braga Quintanilha\",\"placa\":\"ZXC-2298\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Rodrigo Braga Quintanilha Procedimento: sessao de terapia ocupacional Data do atendimento: 29/07/2026 Valor cobrado: R$ 1.660,40"},{"role":"assistant","content":"{\"beneficiario\":\"Rodrigo Braga Quintanilha\",\"procedimento\":\"sessao de terapia ocupacional\",\"valor\":1660.4}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Leonardo Batista Cavalcanti Placa do veiculo: YUI-2216 Data do sinistro: 30/04/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Leonardo Batista Cavalcanti Placa: YUI-2216 Data do atendimento: 30/04/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Leonardo Batista Cavalcanti Placa do veiculo: YUI-2216 Data: 30/04/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Leonardo Batista Cavalcanti Placa do carro: YUI-2216 Data do orcamento: 30/04/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 2.220,80"},{"role":"assistant","content":"{\"segurado\":\"Leonardo Batista Cavalcanti\",\"placa\":\"YUI-2216\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Leonardo Batista Cavalcanti Procedimento: exame de imagem (ressonancia) Data do atendimento: 30/04/2026 Valor cobrado: R$ 2.220,80"},{"role":"assistant","content":"{\"beneficiario\":\"Leonardo Batista Cavalcanti\",\"procedimento\":\"exame de imagem (ressonancia)\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Leonardo Batista Cavalcanti Procedimento realizado: exame de imagem (ressonancia) Data: 30/04/2026 Valor total: R$ 2.220,80"},{"role":"assistant","content":"{\"beneficiario\":\"Leonardo Batista Cavalcanti\",\"procedimento\":\"exame de imagem (ressonancia)\",\"valor\":2220.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Priscila Regina Coutinho Placa do veiculo: WSX-6656 Data do sinistro: 03/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Priscila Regina Coutinho Placa: WSX-6656 Data do atendimento: 03/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Priscila Regina Coutinho Placa do veiculo: WSX-6656 Data: 03/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Priscila Regina Coutinho Placa do carro: WSX-6656 Data do orcamento: 03/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 4.310,00"},{"role":"assistant","content":"{\"segurado\":\"Priscila Regina Coutinho\",\"placa\":\"WSX-6656\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Priscila Regina Coutinho Procedimento: exame de densitometria ossea Data do atendimento: 03/06/2026 Valor cobrado: R$ 4.310,00"},{"role":"assistant","content":"{\"beneficiario\":\"Priscila Regina Coutinho\",\"procedimento\":\"exame de densitometria ossea\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Priscila Regina Coutinho Procedimento realizado: exame de densitometria ossea Data: 03/06/2026 Valor total: R$ 4.310,00"},{"role":"assistant","content":"{\"beneficiario\":\"Priscila Regina Coutinho\",\"procedimento\":\"exame de densitometria ossea\",\"valor\":4310}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Sabrina Rocha Pimentel Placa do veiculo: FDS-8842 Data do sinistro: 08/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Sabrina Rocha Pimentel Placa: FDS-8842 Data do atendimento: 08/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Sabrina Rocha Pimentel Placa do veiculo: FDS-8842 Data: 08/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Sabrina Rocha Pimentel Placa do carro: FDS-8842 Data do orcamento: 08/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 890,00"},{"role":"assistant","content":"{\"segurado\":\"Sabrina Rocha Pimentel\",\"placa\":\"FDS-8842\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Sabrina Rocha Pimentel Procedimento: exame de tomografia Data do atendimento: 08/06/2026 Valor cobrado: R$ 890,00"},{"role":"assistant","content":"{\"beneficiario\":\"Sabrina Rocha Pimentel\",\"procedimento\":\"exame de tomografia\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Sabrina Rocha Pimentel Procedimento realizado: exame de tomografia Data: 08/06/2026 Valor total: R$ 890,00"},{"role":"assistant","content":"{\"beneficiario\":\"Sabrina Rocha Pimentel\",\"procedimento\":\"exame de tomografia\",\"valor\":890}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Mariana dos Santos Pena Placa do veiculo: QWE-1150 Data do sinistro: 24/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Mariana dos Santos Pena Placa: QWE-1150 Data do atendimento: 24/06/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Mariana dos Santos Pena Placa do veiculo: QWE-1150 Data: 24/06/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nOFICINA NOVA ALIANCA LTDA CNPJ 22.333.444/0001-55 Estrada Velha 1200 Segurado: Mariana dos Santos Pena Placa do carro: QWE-1150 Data do orcamento: 24/06/2026 Descricao: reparo de amassado na porta dianteira Valor cobrado: R$ 1.485,70"},{"role":"assistant","content":"{\"segurado\":\"Mariana dos Santos Pena\",\"placa\":\"QWE-1150\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Mariana dos Santos Pena Procedimento: sessao de pilates terapeutico Data do atendimento: 24/06/2026 Valor cobrado: R$ 1.485,70"},{"role":"assistant","content":"{\"beneficiario\":\"Mariana dos Santos Pena\",\"procedimento\":\"sessao de pilates terapeutico\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nHOSPITAL SANTA CLARA CNPJ 66.555.444/0001-22 Rua das Acacias 310 Beneficiario: Mariana dos Santos Pena Procedimento realizado: sessao de pilates terapeutico Data: 24/06/2026 Valor total: R$ 1.485,70"},{"role":"assistant","content":"{\"beneficiario\":\"Mariana dos Santos Pena\",\"procedimento\":\"sessao de pilates terapeutico\",\"valor\":1485.7}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCENTRO MEDICO BEM ESTAR CNPJ 77.888.999/0001-66 Rua da Saude 45 Nome do beneficiario: Mariana dos Santos Pena Procedimento: sessao de pilates terapeutico Data da consulta: 24/06/2026 Valor cobrado: R$ 1.485,70"},{"role":"assistant","content":"{\"beneficiario\":\"Mariana dos Santos Pena\",\"procedimento\":\"sessao de pilates terapeutico\",\"valor\":1485.7}"}]}"""

TEST_JSONL = r"""{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Anderson Machado Freitas Placa do veiculo: WER-4481 Data do sinistro: 01/07/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.390,60"},{"role":"assistant","content":"{\"segurado\":\"Anderson Machado Freitas\",\"placa\":\"WER-4481\",\"valor\":3390.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Debora Martins Cardoso Placa do veiculo: LKM-3376 Data do sinistro: 14/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 4.780,60"},{"role":"assistant","content":"{\"segurado\":\"Debora Martins Cardoso\",\"placa\":\"LKM-3376\",\"valor\":4780.6}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Fabio Vinicius Andrade Pereira Placa do veiculo: UJM-7726 Data do sinistro: 23/06/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 6.310,90"},{"role":"assistant","content":"{\"segurado\":\"Fabio Vinicius Andrade Pereira\",\"placa\":\"UJM-7726\",\"valor\":6310.9}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Vanessa Costa Miranda Placa do veiculo: DFG-7784 Data do sinistro: 27/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.150,80"},{"role":"assistant","content":"{\"segurado\":\"Vanessa Costa Miranda\",\"placa\":\"DFG-7784\",\"valor\":2150.8}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Carolina Almeida Correia Placa do veiculo: QJK-4F82 Data do sinistro: 11/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 2.510,30"},{"role":"assistant","content":"{\"segurado\":\"Carolina Almeida Correia\",\"placa\":\"QJK-4F82\",\"valor\":2510.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Carolina Almeida Correia Placa: QJK-4F82 Data do atendimento: 11/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 2.510,30"},{"role":"assistant","content":"{\"segurado\":\"Carolina Almeida Correia\",\"placa\":\"QJK-4F82\",\"valor\":2510.3}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Felipe Alves Monteiro Placa do veiculo: YHN-5520 Data do sinistro: 18/05/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.450,00"},{"role":"assistant","content":"{\"segurado\":\"Felipe Alves Monteiro\",\"placa\":\"YHN-5520\",\"valor\":3450}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Felipe Alves Monteiro Placa: YHN-5520 Data do atendimento: 18/05/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 3.450,00"},{"role":"assistant","content":"{\"segurado\":\"Felipe Alves Monteiro\",\"placa\":\"YHN-5520\",\"valor\":3450}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Felipe Alves Monteiro Procedimento: consulta de clinica geral Data do atendimento: 18/05/2026 Valor cobrado: R$ 3.450,00"},{"role":"assistant","content":"{\"beneficiario\":\"Felipe Alves Monteiro\",\"procedimento\":\"consulta de clinica geral\",\"valor\":3450}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Amanda Pedro Salgado Placa do veiculo: MJU-6624 Data do sinistro: 17/03/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 1.870,00"},{"role":"assistant","content":"{\"segurado\":\"Amanda Pedro Salgado\",\"placa\":\"MJU-6624\",\"valor\":1870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nAUTO CENTER SILVA - FUNILARIA E PINTURA - CNPJ 98.765.432/0001-11 Av. dos Mecanicos 220 Cliente/Segurado: Amanda Pedro Salgado Placa: MJU-6624 Data do atendimento: 17/03/2026 Servico executado: troca de para-lama e revisao de suspensao dianteira Valor: R$ 1.870,00"},{"role":"assistant","content":"{\"segurado\":\"Amanda Pedro Salgado\",\"placa\":\"MJU-6624\",\"valor\":1870}"}]}
{"messages":[{"role":"user","content":"Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nFUNILARIA RIO BONITO ME CNPJ 45.111.222/0001-33 Rua Rio Bonito 88 Nome do segurado: Amanda Pedro Salgado Placa do veiculo: MJU-6624 Data: 17/03/2026 Orcamento: substituicao de parachoque traseiro e polimento Valor total: R$ 1.870,00"},{"role":"assistant","content":"{\"segurado\":\"Amanda Pedro Salgado\",\"placa\":\"MJU-6624\",\"valor\":1870}"}]}
{"messages":[{"role":"user","content":"Extraia beneficiário, procedimento e valor do recibo médico abaixo.\n\nCLINICA VITALIS SAUDE OCUPACIONAL CNPJ 33.222.111/0001-44 Av. Paulista 900 Paciente/Beneficiario: Amanda Pedro Salgado Procedimento: exame de audiometria Data do atendimento: 17/03/2026 Valor cobrado: R$ 1.870,00"},{"role":"assistant","content":"{\"beneficiario\":\"Amanda Pedro Salgado\",\"procedimento\":\"exame de audiometria\",\"valor\":1870}"}]}"""

def carregar(jsonl_text):
    return [json.loads(linha) for linha in jsonl_text.strip().splitlines()]

exemplos_treino = carregar(TRAIN_JSONL)
exemplos_validacao = carregar(VALID_JSONL)
exemplos_teste = carregar(TEST_JSONL)

print(f"Treino: {len(exemplos_treino)} exemplos")
print(f"Validação: {len(exemplos_validacao)} exemplos")
print(f"Teste: {len(exemplos_teste)} exemplos")
assert len(exemplos_treino) == 157 and len(exemplos_validacao) == 30 and len(exemplos_teste) == 13
print(exemplos_treino[0])


Treino: 157 exemplos
Validação: 30 exemplos
Teste: 13 exemplos
{'messages': [{'role': 'user', 'content': 'Extraia segurado, placa e valor do orçamento de oficina abaixo.\n\nATIVA ORCAMENTOS AUTOMOTIVOS OFICINA ESTRELA LTDA CNPJ 12.345.678/0001-90 Rua das Turbinas 450 Distrito Industrial Segurado: Vinicius Augusto Teixeira Placa do veiculo: YHN-3392 Data do sinistro: 20/02/2026 Descricao do servico: reparo de lataria e pintura no para-choque dianteiro Valor total do reparo: R$ 3.780,90'}, {'role': 'assistant', 'content': '{"segurado":"Vinicius Augusto Teixeira","placa":"YHN-3392","valor":3780.9}'}]}


## Passo 2 - Carregar o modelo completo (sem quantização) e configurar o otimizador de 8 bits

Full fine-tuning treina TODOS os parâmetros - diferente do QLoRA (Módulo 4.2, Colab), não dá pra quantizar os pesos em 4 bits aqui, porque treinar exige atualizar o peso original de verdade, não só um adaptador por cima dele. O jeito real de caber na T4 sem quantizar os pesos é outro: otimizador Adam em 8 bits (`bitsandbytes`, `optim="adamw_8bit"`) - reduz o estado do otimizador de ~8 bytes por parâmetro (Adam padrão, momentos m e v em fp32) pra ~2 bytes por parâmetro - somado a `gradient_checkpointing=True` (recalcula ativações em vez de guardar todas, trocando tempo de GPU por memória) e pesos/gradientes em `bf16` em vez de fp32.


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch_dtype, device_map="auto")

n_params = sum(p.numel() for p in model.parameters())
n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert n_treinaveis == n_params, "full fine-tuning exige cem por cento dos parâmetros treináveis"
assert n_params == MODELOS_VALIDADOS[MODEL_ID]["params_treinaveis"], (
    f"contagem real ({n_params:,}) não bate com a tabela do Passo 0 "
    f"({MODELOS_VALIDADOS[MODEL_ID]['params_treinaveis']:,}) - modelo pode ter mudado no Hub"
)
print(
    f"{n_params:,} parâmetros treináveis, cem por cento do modelo "
    f"(full fine-tuning sempre treina 100% - pra contexto, o LoRA rank 8 usado nos Módulos "
    f"4.2/4.3 desta disciplina, aplicado ao Gemma 4 E2B bem maior, treina só 0,147% dos parâmetros dele)"
)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1,720,574,976 parâmetros treináveis, cem por cento do modelo (full fine-tuning sempre treina 100% - pra contexto, o LoRA rank 8 usado nos Módulos 4.2/4.3 desta disciplina, aplicado ao Gemma 4 E2B bem maior, treina só 0,147% dos parâmetros dele)


## Passo 3 - Treinar de verdade

`SFTTrainer` (biblioteca `trl`), a mesma ferramenta do notebook de LoRA do Módulo 4.2 - a diferença é não passar nenhum `peft_config`: sem adaptador, o `SFTTrainer` atualiza os pesos originais do modelo inteiro. `MAX_STEPS=20` e `LEARNING_RATE=1e-5` (definidos no Passo 0) reproduzem o mesmo orçamento e a mesma taxa de aprendizado do treino real do Módulo 4.4 (MLX, `--fine-tune-type full --iters 20 --learning-rate 1e-5`).


In [6]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

dataset_treino = Dataset.from_list(exemplos_treino)
dataset_validacao = Dataset.from_list(exemplos_validacao)


def collate_fn(exemplos):
    textos = [aplicar_chat_template(tokenizer, ex["messages"], add_generation_prompt=False) for ex in exemplos]
    lote = tokenizer(textos, return_tensors="pt", padding=True, truncation=True, max_length=512)
    rotulos = lote["input_ids"].clone()
    rotulos[rotulos == tokenizer.pad_token_id] = -100
    lote["labels"] = rotulos
    return lote


args = SFTConfig(
    output_dir="qwen-amplitude-full-ft-colab",
    max_length=512,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    optim="adamw_8bit",
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="no",
    eval_strategy="steps",
    eval_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=(torch_dtype == torch.bfloat16),
    lr_scheduler_type="constant",
    report_to="none",
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    seed=0,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset_treino,
    eval_dataset=dataset_validacao,
    processing_class=tokenizer,
    data_collator=collate_fn,
)

resultado = trainer.train()
print(resultado)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,1.444287,1.301716,1.278511,3379.000000,0.750271


TrainOutput(global_step=20, training_loss=1.9745918273925782, metrics={'train_runtime': 42.1337, 'train_samples_per_second': 0.475, 'train_steps_per_second': 0.475, 'total_flos': 28574379313152.0, 'train_loss': 1.9745918273925782, 'epoch': 0.12738853503184713})


## Passo 4 - Testar contra um exemplo difícil de propósito

Mesmo princípio do Passo 5 do notebook de LoRA (Módulo 4.2) e do Passo 3 da Atividade 4 (Módulo 4.4): um exemplo com informação distratora, fora do padrão de treino, pra ver se o modelo aprendeu o padrão de extração ou só decorou o formato dos exemplos vistos.


In [7]:
EXEMPLO_DIFICIL = {
    "instrucao": "Extraia segurado, placa e valor do orçamento de oficina abaixo.",
    "entrada": (
        "OFICINA MECANICA SAO CRISTOVAO CNPJ 09.876.543/0001-21 "
        "Nota: revisao anterior do mesmo veiculo, placa QRS-1122, ja foi paga em 10/02/2026, valor R$ 890,00. "
        "Segurado: Roberta Almeida Castro Placa do veiculo: TUV-4499 "
        "Data do sinistro: 30/06/2026 Descricao do servico: troca de parabrisa "
        "Valor total do reparo: R$ 2.310,75"
    ),
}

texto_usuario = f"{EXEMPLO_DIFICIL['instrucao']}\n\n{EXEMPLO_DIFICIL['entrada']}"
mensagens = [{"role": "user", "content": texto_usuario}]
prompt = aplicar_chat_template(tokenizer, mensagens, add_generation_prompt=True)
entradas = tokenizer(prompt, return_tensors="pt").to(model.device)

saida = model.generate(**entradas, max_new_tokens=100, do_sample=False)
resposta = tokenizer.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True)

print("Exemplo difícil: tem uma placa e um valor de uma revisão ANTERIOR (QRS-1122, R$ 890,00) misturados")
print("no meio do texto, antes do sinistro de verdade (TUV-4499, R$ 2.310,75). Extrair a placa ou o valor")
print("errado, da revisão antiga, é o sintoma de decoreba de posição em vez de entendimento da tarefa.")
print()
print("Resposta do modelo (depois do full fine-tuning):", resposta)


Exemplo difícil: tem uma placa e um valor de uma revisão ANTERIOR (QRS-1122, R$ 890,00) misturados
no meio do texto, antes do sinistro de verdade (TUV-4499, R$ 2.310,75). Extrair a placa ou o valor
errado, da revisão antiga, é o sintoma de decoreba de posição em vez de entendimento da tarefa.

Resposta do modelo (depois do full fine-tuning): {"segurado":"Roberta Almeida Castro","placa":"TUV-4499","valor":2310.75,"data":"30/06/2026","oficinas":["OFICINA MECANICA SAO CRISTOVAO CNPJ 09.876.543/0001-21"],"data_pagamento":"10/02/2026


**Achado real, rodando isso pela primeira vez**: reparem que a resposta acima trouxe campos que não existem no dataset de treino (`data`, `oficinas`, `data_pagamento`) e não fechou o JSON dentro de `max_new_tokens=100` - o modelo aprendeu a estrutura JSON e o padrão de extração central (os três campos certos, distrator ignorado), mas com só 20 passos ainda não travou o formato de saída tão rigidamente quanto o LoRA do Módulo 4.2 trava (adaptador pequeno, mudança cirúrgica sobre um comportamento já restrito). Não é um bug do notebook: é uma diferença real entre LoRA e full fine-tuning de orçamento curto num modelo maior - full fine-tuning muda mais peso de uma vez, então mais passos ajudam a convergir o formato. Quer ver a resposta completa em vez de cortada? Aumentem `max_new_tokens` na célula do Passo 4 pra 200 ou mais e rodem de novo - vale como exercício.


## Passo 5 opcional - Pico real de memória da GPU

`torch.cuda.max_memory_allocated()` depois do treino - útil pra comparar contra a tabela do Passo 0 e contra os números reais do Módulo 4.4 (MLX, pico de 15,338GB pro full fine-tuning do Gemma 4 E2B).


In [8]:
if torch.cuda.is_available():
    pico_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"Pico real de memória de GPU nesta execução: {pico_gb:.2f} GB")
else:
    print("Sem GPU disponível nesta sessão - rode com Runtime > Change runtime type > T4 GPU pra medir de verdade.")


Pico real de memória de GPU nesta execução: 13.51 GB


## Fechamento

Depois de rodar este notebook uma vez com GPU de verdade: anotem o `train_loss` e o `eval_loss` finais do Passo 3, e comparem com os números reais do Módulo 4.4 (MLX, Gemma 4 E2B, val loss final 0,612). Os dois frameworks e os dois modelos não vão bater número a número - arquitetura, tamanho de modelo e otimizador diferem -, mas a pergunta que importa é a mesma dos Módulos 4.3 e 4.4: o modelo aprendeu o padrão de extração, ou só decorou o formato? O exemplo difícil do Passo 4 responde isso, não o número de loss sozinho. E reparem no pico de memória do Passo 5 opcional: mesmo um modelo 2 a 7 vezes menor que o Gemma 4 E2B do curso já usa boa parte dos 16GB da T4 gratuita em full fine-tuning - é a mesma lição do Módulo 4.4 na prática, só que sentida em vez de só lida num número.

---

Ahirton Lopes · Fine-Tuning Toolkit - UNIPDS: Processamento de Dados e Fine-Tuning de Modelos
Prof. Ahirton Lopes, Ph.D. - GDE AI, Microsoft MVP, Senior Manager
